## Section 1 — Project Overview

### Phase : Machine Learning — Feature Adoption & User Churn Prediction

### 1. Project Overview

### Project Title
AI-Powered Feature Adoption Prediction & Causal Impact Analysis for Developer Tools

### Project Objective

The objective of this phase is to develop machine learning models that predict product feature adoption and identify users who may be at risk of churning.

This phase focuses on two predictive analytics problems:

1. Feature Adoption Prediction
2. User Churn Prediction

### Business Goals

- Identify users who are less likely to adopt product features.
- Predict future feature adoption behavior.
- Identify users who may be at risk of churn.
- Understand user behavior and engagement patterns.
- Support targeted onboarding and retention strategies.
- Help product teams make data-driven decisions.

### Technology Stack

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn
- Google BigQuery
- Jupyter Notebook

## Section 2 — Business Problem Definition

## 2. Business Problem Definition

### Problem 1: Feature Adoption Prediction

**Business Problem:**

Some users do not adopt important product features. Product teams need to identify users who are less likely to adopt features so they can provide better onboarding, documentation, and product guidance.

**Proposed Solution:**

Develop a machine learning classification model that predicts whether a user will adopt a selected product feature within a future period.

### Problem 2: User Churn Prediction

**Business Problem:**

Some users or organizations may stop using the platform or allow their subscriptions to expire. Losing these users can negatively affect product growth and recurring revenue.

**Proposed Solution:**

Develop a machine learning classification model that predicts whether an eligible user or organization will meet the project's churn definition within a future period.

### Business Value

These models can help the product team:

- Improve feature adoption.
- Increase user engagement.
- Identify potential churn risks.
- Improve customer retention.
- Prioritize product improvement opportunities.

## Section 3 — Machine Learning Objectives

## 3. Machine Learning Objectives

This project contains two binary classification tasks.

### Model 1: Feature Adoption Prediction

- **Objective:** Predict whether a user will adopt a selected feature.
- **Problem type:** Binary Classification.
- **Output:** Adoption probability and predicted class.

### Model 2: User Churn Prediction

- **Objective:** Predict whether a user or organization will churn.
- **Problem type:** Binary Classification.
- **Output:** Churn probability and predicted class.

### Why Binary Classification?

Both problems have two possible outcomes:

- Positive class (1)
- Negative class (0)

The models will learn patterns from historical user behavior and use those patterns to predict future outcomes.

## Section 4 — Define Target Variables

## 4. Define Target Variables

A target variable is the outcome that a machine learning model attempts to predict.

### 4.1 Feature Adoption Target

Target variable: `feature_adopted`

| Value | Meaning |
|---|---|
| 1 | User adopted the selected feature |
| 0 | User did not adopt the selected feature |

### 4.2 User Churn Target

Target variable: `churned`

| Value | Meaning |
|---|---|
| 1 | User or organization met the defined churn criterion |
| 0 | User or organization did not meet the defined churn criterion |

**Important:** Subscription expiry may be used as a churn proxy if confirmed cancellation data is unavailable. We must clearly document this limitation.

In [3]:
# Define machine learning target variables

ml_targets = {
    "feature_adoption": {
        "target": "feature_adopted",
        "problem_type": "Binary Classification"
    },
    "user_churn": {
        "target": "churned",
        "problem_type": "Binary Classification"
    }
}

for problem, details in ml_targets.items():

    print("\nProblem:", problem)

    for key, value in details.items():
        print(f"{key}: {value}")


Problem: feature_adoption
target: feature_adopted
problem_type: Binary Classification

Problem: user_churn
target: churned
problem_type: Binary Classification


## Section 5 — Independent Variables (Predictors)

## 5. Independent Variables (Predictors)

Independent variables are the input features used by machine learning models to predict the target variable.

### 5.1 Feature Adoption Predictors

| Variable | Description |
|---|---|
| `sdk_integrated` | Whether the user integrated the SDK |
| `first_api_request` | Whether the user made a first API request |
| `total_api_requests` | Total number of API requests |
| `active_days` | Number of active days |
| `previous_feature_adoptions` | Number of previously adopted features |
| `organization_size` | Size of the organization |
| `account_age_days` | Account age in days |

### 5.2 Churn Prediction Predictors

| Variable | Description |
|---|---|
| `total_api_requests` | Total API requests |
| `active_days` | Number of active days |
| `feature_adoption_count` | Number of adopted features |
| `sdk_integrated` | SDK integration status |
| `days_since_last_activity` | Days since the last activity |
| `subscription_plan` | Subscription plan |
| `subscription_age_days` | Subscription age in days |

**Note:** These are candidate predictors. We will verify actual column names and availability in BigQuery before using them. All predictors must be available before the prediction date to prevent data leakage.

## Section 6 — Observation Window and Prediction Window

## 6. Observation Window and Prediction Window

### 6.1 Observation Window

The observation window is the historical period used to calculate model input features.

**Proposed duration:** 30 days.

During this period, we may calculate:

- Total API requests.
- Active days.
- SDK integration status.
- Feature usage.
- User engagement.
- Days since last activity.

### 6.2 Prediction Window

The prediction window is the future period during which the model's target outcome is measured.

**Proposed duration:** 30 days.

### 6.3 Example Timeline

| Period | Purpose |
|---|---|
| Day 1–30 | Observation window |
| Day 31–60 | Prediction window |

For feature adoption, we predict whether the selected feature will be adopted during the prediction window.

For churn, we predict whether the defined churn event will occur during the prediction window.

**Important:** These 30-day windows are initial proposals. We will validate the actual date coverage and business definitions before implementing them.

## Section 7 — Model Evaluation Metrics

## 7. Model Evaluation Metrics

We will use the following metrics to evaluate our machine learning models.

| Metric | Description |
|---|---|
| Accuracy | Percentage of correct predictions |
| Precision | How many predicted positive cases were actually positive |
| Recall | How many actual positive cases the model identified |
| F1-score | Balance between precision and recall |
| ROC-AUC | Ability to distinguish between positive and negative classes |
| Confusion Matrix | Shows correct and incorrect predictions |

### Metric Priorities

**Feature Adoption Prediction:**
- Precision
- Recall
- F1-score
- ROC-AUC

**User Churn Prediction:**
- Recall
- Precision
- F1-score
- ROC-AUC

We will also investigate class imbalance before selecting evaluation metrics and decision thresholds.

## Section 8 — Modeling Dataset Structure

## 8. Modeling Dataset Structure

### Data Grain

Our initial modeling dataset will contain one row per eligible user per prediction date.

### Example Dataset

| user_id | active_days | total_api_requests | sdk_integrated | feature_adopted |
|---|---:|---:|---:|---:|
| U001 | 18 | 450 | 1 | 1 |
| U002 | 7 | 95 | 0 | 0 |
| U003 | 24 | 780 | 1 | 1 |

*Illustrative example only; these are not actual project records.*

### Dataset Requirements

1. Each row must represent a clearly defined user and prediction date.
2. Predictors must come from the observation window.
3. Target variables must come from the prediction window.
4. User IDs should not be used as model predictors.
5. Missing values and duplicate records must be investigated.
6. Training and testing data must be separated carefully.
7. Information from the future must not leak into the predictors.

We may create separate datasets for Feature Adoption Prediction and User Churn Prediction.

## Section 9 — Expected Business Outcomes

## 9. Expected Business Outcomes

### Feature Adoption Prediction

The model should help product teams:

- Identify users who may not adopt important features.
- Prioritize onboarding assistance.
- Improve feature discovery.
- Develop targeted product education campaigns.

### User Churn Prediction

The model should help product teams:

- Identify users or organizations with potential churn risk.
- Prioritize retention campaigns.
- Understand behavioral patterns associated with churn.
- Improve subscription retention strategies.

### Overall Business Value

The predictive models will support data-driven product decisions, improve user activation, and help product teams allocate customer-success resources more effectively.

## Section 10 — Important ML Considerations

## 10. Important Machine Learning Considerations

Before training models, we must consider:

1. **Data Leakage:** Avoid using future information as model input.
2. **Class Imbalance:** Investigate the distribution of positive and negative classes.
3. **Missing Values:** Identify and handle missing data appropriately.
4. **Duplicate Records:** Ensure the dataset has the correct user-date grain.
5. **Target Definition:** Define adoption and churn using verified source data.
6. **Churn Limitation:** Subscription expiry is a proxy unless confirmed cancellation data exists.
7. **Experiment Limitation:** The existing two-row aggregate experiment table is insufficient for user-level causal inference.
8. **Model Generalization:** Evaluate models on data not used for training.

### Important Note

Predictive modeling and causal inference are different tasks. A model that predicts adoption or churn does not establish that a particular product change caused the outcome.

## Section 11 — Project Configuration

In [7]:
# ==========================================
# MACHINE LEARNING PROJECT CONFIGURATION
# ==========================================

PROJECT_NAME = "DevFlow Product Analytics"

# Prediction targets
FEATURE_ADOPTION_TARGET = "feature_adopted"
CHURN_TARGET = "churned"

# Proposed time windows
OBSERVATION_WINDOW_DAYS = 30
PREDICTION_WINDOW_DAYS = 30

# Machine learning problem
PROBLEM_TYPE = "Binary Classification"

# Data grain
DATA_GRAIN = "One row per eligible user per prediction date"

# Evaluation metrics
EVALUATION_METRICS = [
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "roc_auc",
    "confusion_matrix"
]

# Baseline models
BASELINE_MODELS = [
    "Logistic Regression",
    "Decision Tree"
]

# Advanced models
ADVANCED_MODELS = [
    "Random Forest",
    "Gradient Boosting"
]

print("=" * 50)
print("MACHINE LEARNING PROJECT CONFIGURATION")
print("=" * 50)

print("Project:", PROJECT_NAME)
print("Problem:", PROBLEM_TYPE)
print("Data Grain:", DATA_GRAIN)

print("\nPrediction Targets:")
print("1.", FEATURE_ADOPTION_TARGET)
print("2.", CHURN_TARGET)

print("\nObservation Window:", OBSERVATION_WINDOW_DAYS, "days")
print("Prediction Window:", PREDICTION_WINDOW_DAYS, "days")

print("\nEvaluation Metrics:")
for metric in EVALUATION_METRICS:
    print("-", metric)

print("\nBaseline Models:")
for model in BASELINE_MODELS:
    print("-", model)

print("\nAdvanced Models:")
for model in ADVANCED_MODELS:
    print("-", model)

print("\nConfiguration completed successfully!")

MACHINE LEARNING PROJECT CONFIGURATION
Project: DevFlow Product Analytics
Problem: Binary Classification
Data Grain: One row per eligible user per prediction date

Prediction Targets:
1. feature_adopted
2. churned

Observation Window: 30 days
Prediction Window: 30 days

Evaluation Metrics:
- accuracy
- precision
- recall
- f1_score
- roc_auc
- confusion_matrix

Baseline Models:
- Logistic Regression
- Decision Tree

Advanced Models:
- Random Forest
- Gradient Boosting

Configuration completed successfully!


## Section 12 — Final Conclusion

##  S Conclusion

In this step, we defined the machine learning objectives for the DevFlow platform.

Two predictive modeling problems were identified:

1. Feature Adoption Prediction.
2. User Churn Prediction.

We also documented the target variables, candidate predictors, observation and prediction windows, evaluation metrics, modeling dataset structure, and expected business outcomes.

The next step is to inspect the underlying BigQuery source tables and prepare a user-level dataset for machine learning.

